# aule — Spectral & Earth Observation metrics

This notebook covers two related families:

- **Spectral metrics** (`aule.metrics.spectral`): error measured in frequency space (FFT), useful for spotting artifacts that pixel-wise metrics miss (e.g. checkerboard patterns from upsampling, blurring).
- **Earth observation metrics** (`aule.metrics.earth_observation`): metrics specific to remote sensing, like normalized difference indices (NDVI, NDWI, ...) and multi-temporal change detection.

In [ ]:
!pip install aule

## Setup

In [2]:
import numpy as np

np.random.seed(0)

gt   = np.random.rand(64, 64, 1)
pred = gt + np.random.normal(0, 0.05, gt.shape)

## Spectral error (2D FFT)

`spectral_error` compares the 2D power spectra directly; it's sensitive to *where* (which frequency/direction) the model gets the spectrum wrong.

In [3]:
from aule.metrics import spectral_error

print('Spectral error:', spectral_error(gt, pred))
print('Spectral error (identical):', spectral_error(gt, gt))  # should be ~0

Spectral error: 0.0004365603437792125
Spectral error (identical): 0.0


## Radially-averaged PSD error

`psd_radial_error` is often more robust: it averages the power spectrum over all directions at each frequency magnitude, summarizing the spectrum as a 1D profile before comparing. Less sensitive to orientation artifacts than the raw 2D version above.

In [4]:
from aule.metrics import psd_radial_error

print('PSD radial error:', psd_radial_error(gt, pred))

PSD radial error: 9.800655382109811e-05


## Gradient (edge) error

`gradient_error` uses Sobel filters to compare spatial gradients rather than raw values — useful when sharp transitions matter (coastlines, fronts, urban boundaries) more than smooth interior areas.

In [5]:
from aule.metrics import gradient_error

print('Gradient error (L1):', gradient_error(gt, pred, norm='l1'))
print('Gradient error (L2):', gradient_error(gt, pred, norm='l2'))

Gradient error (L1): 0.13834182673748924
Gradient error (L2): 0.030217426996953595


## Spectral Angle Mapper (multi-band)

`spectral_angle_mapper` (SAM) is a classic hyperspectral/multispectral metric: it measures the angle between the per-pixel spectral signature vectors of ground truth and prediction, ignoring overall brightness. Requires multiple channels (bands).

In [6]:
# simulate a 4-band image (e.g. RGB + NIR)
gt_multiband   = np.random.rand(32, 32, 4) + 0.2
pred_multiband = gt_multiband * 1.05  # a pure brightness scaling: SAM should be near 0

from aule.metrics import spectral_angle_mapper

print('SAM (deg), brightness-scaled prediction:', spectral_angle_mapper(gt_multiband, pred_multiband))

pred_multiband_noisy = gt_multiband + np.random.normal(0, 0.1, gt_multiband.shape)
print('SAM (deg), noisy prediction:            ', spectral_angle_mapper(gt_multiband, pred_multiband_noisy))

SAM (deg), brightness-scaled prediction: 2.4718280065094473e-07
SAM (deg), noisy prediction:             6.394129160246132


## Normalized difference indices (NDVI, NDWI, ...)

`normalized_difference_index` implements the generic formula `(band_a - band_b) / (band_a + band_b)`, which covers NDVI (NIR, Red), NDWI (Green, NIR), NDSI (Green, SWIR), and similar two-band indices.

In [7]:
from aule.metrics import normalized_difference_index

# simulate NIR and Red bands
nir = np.random.rand(64, 64, 1) * 0.5 + 0.3
red = np.random.rand(64, 64, 1) * 0.3 + 0.1

ndvi = normalized_difference_index(nir, red)
# note: the output is always in the canonical (batch, H, W, C, T) shape
print('NDVI shape (canonical batch/H/W/C/T):', ndvi.shape)
print('NDVI range: [{:.3f}, {:.3f}]'.format(ndvi.min(), ndvi.max()))

NDVI shape (canonical batch/H/W/C/T): (1, 64, 64, 1, 1)
NDVI range: [-0.132, 0.773]


## Index error

If the downstream application cares about the derived index (e.g. NDVI) rather than the raw bands, `index_error` computes the RMSE directly on the index, comparing ground-truth-derived vs prediction-derived index values.

In [8]:
from aule.metrics import index_error

pred_nir = nir + np.random.normal(0, 0.02, nir.shape)
pred_red = red + np.random.normal(0, 0.02, red.shape)

score = index_error(nir, red, pred_nir, pred_red)
print('NDVI RMSE:', score)

NDVI RMSE: 0.04200173287048886


## Change detection error

When a time axis is present, `change_detection_error` measures how well the model reproduces *changes between time steps* (rather than absolute values) — important for monitoring tasks like deforestation or urban growth detection.

In [9]:
# 12 monthly time steps; note data_format='hwct' is required since a time axis is present
gt_series   = np.random.rand(32, 32, 1, 12)
pred_series = gt_series + np.random.normal(0, 0.05, gt_series.shape)

from aule.metrics import change_detection_error

score = change_detection_error(gt_series, pred_series, data_format='hwct')
print('Change detection error:', score)

Change detection error: 0.07027945948203834
